# Rubrique 1 — Calcul numérique et simulation

```{admonition} Objectifs de cette rubrique
:class: tip

À la fin de cette rubrique, vous serez capable de :
- Construire et manipuler des tableaux NumPy N-dimensionnels
- Vectoriser des calculs physiques sans boucles Python explicites
- Diagonaliser des Hamiltoniens par appel à LAPACK via NumPy
- Intégrer des EDO par méthodes adaptatives (DOP853, Radau)
- Implémenter une simulation Monte-Carlo de Metropolis-Hastings
```

---

## 1.1 Positionnement épistémique

Le calcul numérique en Python repose sur un principe architectural unique : 
**le code Python orchestre, le code compilé exécute.**

```{figure} ../../_static/architecture_numpy.png
:name: architecture-numpy
:width: 70%
:align: center

Architecture de la pile NumPy/SciPy : Python comme orchestrateur,
C/Fortran comme moteur de calcul.
```

NumPy {cite}`harris2020array` et SciPy {cite}`virtanen2020scipy` sont des 
interfaces Python sur des noyaux C/Fortran (BLAS, LAPACK, QUADPACK).

```{warning}
**Règle fondamentale** : toute boucle Python explicite sur un tableau 
numérique est une erreur de conception, sauf si elle est physiquement 
irréductible (dépendance séquentielle entre itérations).
```


In [ ]:
# Importations standard — cellule à exécuter en premier
import numpy as np
import scipy
import matplotlib.pyplot as plt
from matplotlib import rcParams

# Configuration matplotlib pour figures de qualité
rcParams.update({
    'font.size'       : 10,
    'axes.labelsize'  : 10,
    'figure.dpi'      : 120,
    'axes.spines.top' : True,
    'axes.spines.right': True,
})

print(f'NumPy  : {np.__version__}')
print(f'SciPy  : {scipy.__version__}')

## 1.2 NumPy : fondements

### 1.2.1 Le tableau N-dimensionnel (`ndarray`)

```{admonition} Problème physique
:class: dropdown

En physique, toute grandeur discrétisée — une fonction d'onde sur un maillage 
spatial, un champ électromagnétique sur une grille, une série temporelle de 
mesures — se représente naturellement comme un tableau multidimensionnel.

**Question computationnelle** : comment stocker et manipuler ces structures 
de façon à ce que les opérations algébriques s'exécutent au coût minimal ?
```


In [ ]:
# ── Création de tableaux ─────────────────────────────────
# Grille uniforme sur [0, 2π], 1000 points
x   = np.linspace(0, 2 * np.pi, 1000)  # dtype float64 par défaut

# Fonction d'onde d'une onde plane : ψ(x) = exp(ikx)
k   = 1.0
psi = np.exp(1j * k * x)               # dtype complex128 automatiquement

print(f'Forme    : {psi.shape}')
print(f'Type     : {psi.dtype}')
print(f'Strides  : {psi.strides}  (16 octets par complex128)')
print(f'Mémoire  : {psi.nbytes / 1024:.1f} Ko')

### 1.2.2 Broadcasting

```{admonition} Règle de broadcasting
:class: note

Deux dimensions sont compatibles si elles sont **égales**, ou si l'une d'elles 
vaut **1**. NumPy étend les dimensions de taille 1 sans allocation mémoire.
```


In [ ]:
import time

omega = 1.0
x_1d  = np.linspace(-5, 5, 500)
y_1d  = np.linspace(-5, 5, 500)

# Broadcasting : x[:, None] (500,1) × y[None, :] (1,500) → (500,500)
t0 = time.perf_counter()
V  = 0.5 * omega**2 * (x_1d[:, None]**2 + y_1d[None, :]**2)
t1 = time.perf_counter()

print(f'V shape          : {V.shape}')
print(f'Temps (broadcast): {(t1-t0)*1000:.3f} ms')
print(f'V minimum        : {V.min():.4f}  (attendu : 0.0)')